In [1]:
# ── 1) Drive 연결 ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2) CSV 자동 탐색 ─────────────────────────────────────────
import glob, os
hits = glob.glob('/content/drive/MyDrive/**/GSE103322_WBP5_cell_level_raw.csv',
                 recursive=True)
print("찾은 경로:", hits)

if not hits:
    print("\n못 찾았습니다. WBP5 관련 폴더 목록:")
    for p in glob.glob('/content/drive/MyDrive/**/GSE103322*', recursive=True)[:30]:
        print("  ", p)
    raise SystemExit("위 목록을 알려주십시오.")

CSV = hits[0]

# ── 3) 계산 ──────────────────────────────────────────────────
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

df = pd.read_csv(CSV, encoding='utf-8-sig')
df.columns = [c.strip() for c in df.columns]
df['WBP5'] = pd.to_numeric(df['WBP5'], errors='coerce')
bad = df['WBP5'].isna().sum()
if bad: print(f"※ WBP5가 수치가 아닌 행 {bad}개 제외")
df = df.dropna(subset=['WBP5'])
print(f"{len(df):,} 행 | is_cancer {df.is_cancer.value_counts().to_dict()}"
      f" | site {df.site.value_counts().to_dict()}")

def stats(a, b, na, nb):
    a, b = np.asarray(a, float), np.asarray(b, float)
    gt = int((a[:,None] > b[None,:]).sum())
    lt = int((a[:,None] < b[None,:]).sum())
    return dict(name=f"{na} vs. {nb}", n_a=len(a), n_b=len(b),
                mean_a=a.mean(), mean_b=b.mean(),
                pct_a=(a>0).mean()*100, pct_b=(b>0).mean()*100,
                fc=a.mean()/b.mean(),                    # FC = mean_A / mean_B
                p=mannwhitneyu(a, b, alternative='two-sided')[1],
                delta=(gt-lt)/(len(a)*len(b)))

r1 = stats(df.loc[df.is_cancer==1,'WBP5'], df.loc[df.is_cancer==0,'WBP5'],
           'Cancer', 'Non-cancer')
m = df[df.is_cancer==1]
r2 = stats(m.loc[m.site=='Primary','WBP5'], m.loc[m.site=='Lymph node','WBP5'],
           'Primary (cancer)', 'LN (cancer)')

for r in (r1, r2):
    print(f"\n{r['name']}   n = {r['n_a']:,} / {r['n_b']:,}")
    print(f"  mean    {r['mean_a']:.4f} / {r['mean_b']:.4f}")
    print(f"  PCT+    {r['pct_a']:.2f}% / {r['pct_b']:.2f}%")
    print(f"  FC      {r['fc']:.4f}")
    print(f"  MWU p   {r['p']:.4e}")
    print(f"  Cliff δ {r['delta']:+.4f}")

# ── 4) Row 1 정본 검증 ───────────────────────────────────────
exp = [('n_A',r1['n_a'],2215,0), ('n_B',r1['n_b'],3687,0),
       ('mean_A',r1['mean_a'],2.3813,.005), ('mean_B',r1['mean_b'],1.3231,.005),
       ('PCT+_A',r1['pct_a'],68.89,.05), ('PCT+_B',r1['pct_b'],28.97,.05),
       ('FC',r1['fc'],1.80,.005), ('delta',r1['delta'],0.308,.005)]
print("\n[Row 1 정본 검증]")
ok_all = True
for lab, got, e, tol in exp:
    ok = abs(got-e) <= tol; ok_all &= ok
    print(f"  {'OK    ' if ok else '불일치'}  {lab:<7} {got:>10.4f}  (기대 {e})")

print("\n" + ("=> Row 2 FC = %.2f  ← Table 1 정답" % r2['fc'] if ok_all
              else "=> Row 1 불일치. 이 CSV는 정본이 아닐 수 있습니다."))

# ── 5) 결과 저장 (Colab 좌측 파일창에서 다운로드) ────────────
pd.DataFrame([{
    'Contrast': r['name'], 'n_A': r['n_a'], 'n_B': r['n_b'],
    'mean_A': round(r['mean_a'],4), 'mean_B': round(r['mean_b'],4),
    'PCT+_A (%)': round(r['pct_a'],2), 'PCT+_B (%)': round(r['pct_b'],2),
    'FC (mean_A/mean_B)': round(r['fc'],4), 'MWU_p': r['p'],
    'Cliffs_delta': round(r['delta'],4),
} for r in (r1, r2)]).to_csv('WBP5_Table1_results_v2.csv',
                             index=False, encoding='utf-8-sig')
print("저장: WBP5_Table1_results_v2.csv")

Mounted at /content/drive
찾은 경로: ['/content/drive/MyDrive/WBP5 /GSE103322/GSE103322_WBP5_cell_level_raw.csv']
5,902 행 | is_cancer {0: 3687, 1: 2215} | site {'Primary': 4541, 'Lymph node': 1361}

Cancer vs. Non-cancer   n = 2,215 / 3,687
  mean    2.3813 / 1.3231
  PCT+    68.89% / 28.97%
  FC      1.7998
  MWU p   1.3268e-105
  Cliff δ +0.3075

Primary (cancer) vs. LN (cancer)   n = 1,427 / 788
  mean    2.4629 / 2.2334
  PCT+    71.34% / 64.47%
  FC      1.1028
  MWU p   6.1310e-03
  Cliff δ +0.0692

[Row 1 정본 검증]
  OK      n_A      2215.0000  (기대 2215)
  OK      n_B      3687.0000  (기대 3687)
  OK      mean_A      2.3813  (기대 2.3813)
  OK      mean_B      1.3231  (기대 1.3231)
  OK      PCT+_A     68.8939  (기대 68.89)
  OK      PCT+_B     28.9666  (기대 28.97)
  OK      FC          1.7998  (기대 1.8)
  OK      delta       0.3075  (기대 0.308)

=> Row 2 FC = 1.10  ← Table 1 정답
저장: WBP5_Table1_results_v2.csv


In [2]:
# ── Table 2 재계산: WBP5 vs p-EMT 마커 Spearman 상관 ──────────
import glob, numpy as np, pandas as pd
from scipy.stats import spearmanr

hits = glob.glob('/content/drive/MyDrive/**/GSE103322_WBP5_EMT_markers_cell_level.csv',
                 recursive=True)
print("찾은 경로:", hits)
if not hits:
    raise SystemExit("파일을 찾지 못했습니다.")

d = pd.read_csv(hits[0], encoding='utf-8-sig')
d.columns = [c.strip() for c in d.columns]
print(f"{len(d):,} 행")
print("열 목록:", list(d.columns))

찾은 경로: ['/content/drive/MyDrive/WBP5 /GSE103322/GSE103322_WBP5_EMT_markers_cell_level.csv']
5,902 행
열 목록: ['cell_id', 'WBP5', 'ITGA5', 'LAMC2', 'VIM', 'KRT14', 'KRT17', 'is_cancer', 'site', 'non_cancer_type', 'compartment', 'patient', 'WBP5_detected', 'ITGA5_detected', 'LAMC2_detected', 'VIM_detected', 'KRT14_detected', 'KRT17_detected']


In [3]:
# ── Table 2 재계산: WBP5 vs p-EMT/invasion 마커 Spearman ρ ────
import numpy as np, pandas as pd
from scipy.stats import spearmanr

CSV = '/content/drive/MyDrive/WBP5 /GSE103322/GSE103322_WBP5_EMT_markers_cell_level.csv'
d = pd.read_csv(CSV, encoding='utf-8-sig')
d.columns = [c.strip() for c in d.columns]

MARKERS = ['LAMC2', 'ITGA5', 'KRT14', 'VIM', 'KRT17']
PAPER   = {'LAMC2': 0.169, 'ITGA5': 0.169, 'KRT14': 0.125,
           'VIM': 0.120, 'KRT17': 0.002}          # 논문 Table 2

for col in ['WBP5'] + MARKERS:
    d[col] = pd.to_numeric(d[col], errors='coerce')

canc = d[d.is_cancer == 1].dropna(subset=['WBP5'] + MARKERS)
print(f"악성 세포 n = {len(canc):,}  (논문 기재값 2,215)\n")

print(f"{'Marker':<8}{'rho':>9}{'p':>13}{'논문':>9}{'차이':>9}   판정")
print("-" * 58)
rows = []
for mk in MARKERS:
    rho, p = spearmanr(canc['WBP5'], canc[mk])
    diff = rho - PAPER[mk]
    ok = abs(diff) < 0.005
    print(f"{mk:<8}{rho:>9.4f}{p:>13.2e}{PAPER[mk]:>9.3f}{diff:>+9.4f}   "
          f"{'OK' if ok else '불일치'}")
    rows.append({'Marker': mk, 'Spearman_rho': round(rho, 4),
                 'p_value': p, 'n': len(canc)})

# 참고: 전체 세포 기준 (악성 한정이 아닌 경우와 비교)
print(f"\n[참고] 전체 세포 n = {len(d):,} 기준")
for mk in MARKERS:
    sub = d.dropna(subset=['WBP5', mk])
    rho, p = spearmanr(sub['WBP5'], sub[mk])
    print(f"  {mk:<8}{rho:>9.4f}{p:>13.2e}")

pd.DataFrame(rows).to_csv('WBP5_Table2_results_v2.csv',
                          index=False, encoding='utf-8-sig')
print("\n저장: WBP5_Table2_results_v2.csv")

악성 세포 n = 2,215  (논문 기재값 2,215)

Marker        rho            p       논문       차이   판정
----------------------------------------------------------
LAMC2      0.1689     1.24e-15    0.169  -0.0001   OK
ITGA5      0.1688     1.27e-15    0.169  -0.0002   OK
KRT14      0.1254     3.16e-09    0.125  +0.0004   OK
VIM        0.1203     1.35e-08    0.120  +0.0003   OK
KRT17      0.0019     9.27e-01    0.002  -0.0001   OK

[참고] 전체 세포 n = 5,902 기준
  LAMC2      0.2146     1.85e-62
  ITGA5      0.2855    3.99e-111
  KRT14      0.2806    3.43e-107
  VIM        0.0307     1.82e-02
  KRT17      0.2647     3.52e-95

저장: WBP5_Table2_results_v2.csv


In [4]:
# ── EGFR / PTPRC / EPCAM 추출 후 상관 계산 ───────────────────
import gzip, numpy as np, pandas as pd
from scipy.stats import spearmanr

GZ  = '/content/drive/MyDrive/WBP5 /SCRNA_WORK/GSE103322/GSE103322_HNSCC_all_data.txt.gz'
REF = '/content/drive/MyDrive/WBP5 /GSE103322/GSE103322_WBP5_EMT_markers_cell_level.csv'
WANT = {'WBP5', 'EGFR', 'PTPRC', 'EPCAM'}

with gzip.open(GZ, 'rt') as f:
    cells = f.readline().rstrip('\n').split('\t')[1:]
    for _ in range(5):
        f.readline()
    found = {}
    for line in f:
        p = line.rstrip('\n').split('\t')
        g = p[0].strip().strip("'").strip('"')
        if g in WANT and g not in found:
            found[g] = np.array(p[1:len(cells)+1], dtype=np.float32)
            if len(found) == len(WANT):
                break

print("추출된 유전자:", sorted(found))
missing = WANT - set(found)
if missing:
    print("※ 파일에서 못 찾은 유전자:", missing)

expr = pd.DataFrame(found, index=cells)

ref = pd.read_csv(REF, encoding='utf-8-sig')
ref.columns = [c.strip() for c in ref.columns]
canc_ids = ref.loc[ref.is_cancer == 1, 'cell_id']
canc_ids = [c for c in canc_ids if c in expr.index]
sub = expr.loc[canc_ids]
print(f"\n악성 세포 n = {len(sub):,}")

PAPER = {'EGFR': 0.077, 'PTPRC': 0.015, 'EPCAM': -0.081}
print(f"\n{'Marker':<8}{'rho':>9}{'p':>13}{'논문':>9}{'차이':>9}")
print("-" * 48)
for mk in ['EGFR', 'PTPRC', 'EPCAM']:
    if mk not in sub:
        continue
    rho, p = spearmanr(sub['WBP5'], sub[mk])
    print(f"{mk:<8}{rho:>9.4f}{p:>13.2e}{PAPER[mk]:>9.3f}{rho-PAPER[mk]:>+9.4f}")

# 교차 검증: WBP5가 기존 CSV와 동일한지
chk = ref.set_index('cell_id').loc[canc_ids, 'WBP5']
print(f"\nWBP5 일치 검증 (최대 절대차): "
      f"{np.abs(sub['WBP5'].values - chk.values).max():.6f}")

추출된 유전자: ['EGFR', 'EPCAM', 'PTPRC', 'WBP5']

악성 세포 n = 2,215

Marker        rho            p       논문       차이
------------------------------------------------
EGFR       0.0766     3.08e-04    0.077  -0.0004
PTPRC      0.0150     4.81e-01    0.015  -0.0000
EPCAM     -0.0814     1.26e-04   -0.081  -0.0004

WBP5 일치 검증 (최대 절대차): 0.000000


In [5]:
import shutil, os
dst = '/content/drive/MyDrive/WBP5 /GSE103322'
for f in ['WBP5_Table2_results_v2.csv']:
    if os.path.exists(f):
        shutil.copy(f, dst); print("복사:", f)

복사: WBP5_Table2_results_v2.csv
